# ubctgdb examples

Run the cells in order. `example_prices` is a ready-made table with three fictional prices.
The upload examples create or replace the shared practice table `example_prices_copy`.

Install with `%pip install --upgrade git+https://github.com/UBC-Trading-Group/ubctgdb.git@r2-storage` and restart the notebook kernel. Save `.env` beside this notebook:

```dotenv
R2_ENDPOINT_URL=https://YOUR_ACCOUNT_ID.r2.cloudflarestorage.com
R2_ACCESS_KEY_ID=your_access_key
R2_SECRET_ACCESS_KEY=your_secret_key
R2_BUCKET=ubctg-data
YOUR_NAME=Alex
```

In [1]:
import ubctgdb as db

## List tables
Shows UTC update times to the minute and file sizes in B, KB, MB or GB.

In [2]:
print(db.list_tables())

            table  rows        updated_at     size updated_by
0  example_prices     3  2026-09-25 21:53  1.75 KB       Alex


## Describe a table

In [3]:
print(db.describe("example_prices")["schema"])

{'symbol': 'large_string', 'price': 'double'}


## Preview a table

In [4]:
print(db.preview("example_prices").to_string(index=False))

symbol  price
   AAA   10.5
   BBB   20.0
   CCC   30.5


## Read into pandas

In [5]:
df = db.read_table("example_prices")
print(df.to_string(index=False))

symbol  price
   AAA   10.5
   BBB   20.0
   CCC   30.5


## Download a file

In [6]:
path = db.download_table("example_prices", "example_prices.parquet", overwrite=True)
print(path.name)

example_prices.parquet


## Upload a DataFrame

In [7]:
result = db.upload_dataframe(
    df,
    table="example_prices_copy",
    description="Three fictional stock prices for practice.",
    replace_table=True,
)
print(result["table"], result["rows"])
print(db.describe("example_prices_copy")["description"])

example_prices_copy 3
Three fictional stock prices for practice.


## Upload a Parquet file

In [8]:
result = db.upload_parquet("example_prices.parquet", table="example_prices_copy", replace_table=True)
print(result["table"], result["rows"])
print(db.describe("example_prices_copy")["description"])

example_prices_copy 3
Three fictional stock prices for practice.


## Rename a table
The new name must be unused. This renames only the practice copy.

In [9]:
print(db.rename_table("example_prices_copy", "example_prices_renamed"))

{'old_name': 'example_prices_copy', 'new_name': 'example_prices_renamed'}


## Delete a table
Permanently remove the renamed practice copy. The original example_prices stays available.

In [10]:
print(db.delete_table("example_prices_renamed"))

{'table': 'example_prices_renamed', 'deleted': True}


## Caching
Repeated reads reuse the local file after checking R2 for changes. Use refresh=True to download again. The cache holds up to 10 GB across notebooks and buckets; least recently used files are removed first. Oversized files are not retained. Internet access is needed for the freshness check. Clearing the cache only removes local files.

In [11]:
df = db.read_table("example_prices")
df = db.read_table("example_prices", refresh=True)
print(db.cache_info().to_string(index=False))
print("Cleared:", db.clear_cache("example_prices"))
print("Cleared for this bucket:", db.clear_cache())

         table  size_bytes                           last_used
example_prices        1748 2026-09-25 21:53:17.603689194+00:00
Cleared: 1
Cleared for this bucket: 0


## Folders
Use / in table names; no separate folder creation is needed. folder= includes all subfolders. This example creates the shared practice/prices table.

In [12]:
db.upload_dataframe(df, table="practice/prices", replace_table=True)
df = db.read_table("practice/prices")
print(db.list_tables())
print("Practice folder:")
print(db.list_tables(folder="practice"))

             table  rows        updated_at     size updated_by
0  practice/prices     3  2026-09-25 21:53  1.75 KB       Alex
1   example_prices     3  2026-09-25 21:53  1.75 KB       Alex
Practice folder:
             table  rows        updated_at     size updated_by
0  practice/prices     3  2026-09-25 21:53  1.75 KB       Alex


Use your own table name for real work. `replace_table=True` replaces existing data with no undo.
The downloaded `example_prices.parquet` stays beside this notebook.